In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training and testing datasets
print("Training Data:")
display(train_df.head())
print("Testing Data:")
display(test_df.head())

# Check the information of the datasets
print("Training Data Info:")
train_df.info()
print("Testing Data Info:")
test_df.info()

# Check for missing values
print("Missing values in Training Data:")
print(train_df.isnull().sum())
print("Missing values in Testing Data:")
print(test_df.isnull().sum())

# Summary statistics
print("Summary Statistics for Training Data:")
display(train_df.describe(include='all'))

# Distribution of the target variable
sns.countplot(x='outcome', data=train_df)
plt.title('Distribution of Outcome in Training Data')
plt.show()

# Correlation matrix for numerical features
numeric_features = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_features].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix for Numerical Features')
plt.show()


Training Data:


,surgery,hospital_number,rectal_temp,pulse,respiratory_rate,peripheral_pulse,mucous_membrane,capillary_refill_time,outcome
0,yes,527706,39.0,84.0,24.0,normal,bright_pink,less_3_sec,died
1,yes,528641,38.5,66.0,21.0,normal,bright_pink,less_3_sec,lived
2,yes,535043,37.3,72.0,30.0,reduced,dark_cyanotic,more_3_sec,euthanized
3,yes,535043,38.1,84.0,66.0,reduced,pale_cyanotic,less_3_sec,euthanized
4,yes,528890,39.0,60.0,24.0,reduced,dark_cyanotic,more_3_sec,died


Testing Data:


,surgery,hospital_number,rectal_temp,pulse,respiratory_rate,peripheral_pulse,mucous_membrane,capillary_refill_time,outcome
0,no,535381,39.4,86.0,21.0,normal,pale_pink,less_3_sec,euthanized
1,yes,535029,37.5,112.0,12.0,normal,bright_pink,less_3_sec,euthanized
2,yes,529461,38.5,72.0,44.0,reduced,bright_red,more_3_sec,died
3,yes,534157,38.4,40.0,16.0,reduced,pale_pink,less_3_sec,euthanized
4,yes,529777,38.9,40.0,24.0,normal,pale_pink,less_3_sec,lived


Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                986 non-null    object 
 1   hospital_number        986 non-null    int64  
 2   rectal_temp            986 non-null    float64
 3   pulse                  986 non-null    float64
 4   respiratory_rate       986 non-null    float64
 5   peripheral_pulse       938 non-null    object 
 6   mucous_membrane        971 non-null    object 
 7   capillary_refill_time  982 non-null    object 
 8   outcome                986 non-null    object 
dtypes: float64(3), int64(1), object(5)
memory usage: 69.5+ KB
Testing Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery  

,surgery,hospital_number,rectal_temp,pulse,respiratory_rate,peripheral_pulse,mucous_membrane,capillary_refill_time,outcome
count,986,9.860000e+02,986.000000,986.000000,986.000000,938,971,982,986
unique,2,NaN,NaN,NaN,NaN,4,6,2,3
top,yes,NaN,NaN,NaN,NaN,reduced,pale_pink,less_3_sec,lived
freq,705,NaN,NaN,NaN,NaN,580,232,677,452
mean,NaN,9.746363e+05,38.186207,79.477688,30.056795,NaN,NaN,NaN,NaN
std,NaN,1.385090e+06,0.774303,29.168581,16.318555,NaN,NaN,NaN,NaN
min,NaN,5.213990e+05,35.400000,30.000000,8.000000,NaN,NaN,NaN,NaN
25%,NaN,5.288010e+05,37.800000,52.500000,18.000000,NaN,NaN,NaN,NaN
50%,NaN,5.297960e+05,38.100000,76.000000,28.000000,NaN,NaN,NaN,NaN
75%,NaN,5.341450e+05,38.500000,99.500000,36.000000,NaN,NaN,NaN,NaN


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the train_df DataFrame from the finished tasks
column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-31 12:37:21.052 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'outcome'], 'Numeric': ['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Copy the datasets to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Handling missing values
fill_missing = FillMissingValue(features=['peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'], strategy='most_frequent')
train_df_copy = fill_missing.fit_transform(train_df_copy)
test_df_copy = fill_missing.transform(test_df_copy)

# Encoding categorical variables
label_encode = LabelEncode(features=['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'])
train_df_copy = label_encode.fit_transform(train_df_copy)
test_df_copy = label_encode.transform(test_df_copy)

# Scaling numerical features
scale_features = StandardScale(features=['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'])
train_df_copy = scale_features.fit_transform(train_df_copy)
test_df_copy = scale_features.transform(test_df_copy)

# Display the first few rows of the preprocessed datasets
print("Preprocessed Training Data:")
display(train_df_copy.head())
print("Preprocessed Testing Data:")
display(test_df_copy.head())


Preprocessed Training Data:


,surgery,hospital_number,rectal_temp,pulse,respiratory_rate,peripheral_pulse,mucous_membrane,capillary_refill_time,outcome
0,2,-0.322836,1.051534,0.155119,-0.371348,2,0,0,died
1,2,-0.322161,0.405464,-0.462296,-0.555281,2,0,0,lived
2,2,-0.317536,-1.145102,-0.256491,-0.003482,3,2,1,euthanized
3,2,-0.317536,-0.111391,0.155119,2.203715,3,4,0,euthanized
4,2,-0.321981,1.051534,-0.668101,-0.371348,3,2,1,died


Preprocessed Testing Data:


,surgery,hospital_number,rectal_temp,pulse,respiratory_rate,peripheral_pulse,mucous_membrane,capillary_refill_time,outcome
0,0,-0.317292,1.568389,0.223721,-0.555281,2,5,0,euthanized
1,2,-0.317546,-0.886675,1.115543,-1.107081,2,0,0,euthanized
2,2,-0.321568,0.405464,-0.256491,0.854872,3,1,1,died
3,2,-0.318176,0.276250,-1.354119,-0.861837,3,5,0,euthanized
4,2,-0.321340,0.922320,-1.354119,-0.371348,2,5,0,lived


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the preprocessed training data DataFrame
column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': ['outcome'], 'Numeric': ['surgery', 'hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

# Split the preprocessed training data into features and target
X_train = train_df_copy.drop(columns=['outcome'])
y_train = train_df_copy['outcome']

# Split the preprocessed testing data into features and target
X_test = test_df_copy.drop(columns=['outcome'])
y_test = test_df_copy['outcome']

# Initialize the XGBoost classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Perform grid search to find the best parameters
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Predict on the test set
y_pred = best_model.predict(X_test)

# Calculate the F1 score
f1 = f1_score(y_test, y_pred, average='macro')

# Print the classification report
print(classification_report(y_test, y_pred))

# Print the F1 score
print(f"Test Set F1 Score: {f1:.4f}")


ValueError: 
All the 324 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
324 fits failed with the following error:
Traceback (most recent call last):
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\sklearn.py", line 1491, in fit
    raise ValueError(
ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got ['died' 'euthanized' 'lived']
